# 07.7 - Attention for NLP

**Phase:** 07 - NLP

**Status:** VERIFIED

---
## 1. What Are We Solving?

A fixed-length hidden state must compress an entire sentence into one vector (bottleneck). **Attention** lets each position dynamically focus on other positions by computing a relevance-weighted sum — the key innovation behind transformers.

## 2. Why Does This Matter?

When translating, you glance back at relevant source words as you produce each target word. Attention formalizes this 'glancing back'. It removes the fixed-vector bottleneck and is the conceptual bridge from RNNs to transformers (Phase 08).

## 3. Prerequisites

- Unit 07.6 (sequence models)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement dot-product attention from scratch
- Explain query / key / value and attention weights
- Scale by sqrt(d_k) to avoid softmax saturation
- Visualize and interpret attention weights

## 5. Mental Model

```text
No attention:  encoder -> single vector -> decoder
With attention: encoder -> all states -> weighted access -> decoder
Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) @ V
```
Query asks 'how relevant is each key?'; weights choose which values to emphasize.


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Scaled Dot-Product Attention From Scratch


In [2]:
def scaled_dot_attention(Q, K, V, mask=None):
    # Q: (B, Tq, d), K: (B, Tk, d), V: (B, Tk, d)
    d = K.shape[-1]
    scores = torch.bmm(Q, K.transpose(1, 2)) / (d ** 0.5)  # scale by sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    out = torch.bmm(weights, V)
    return out, weights

Q = torch.randn(2, 3, 8)  # batch=2, query_len=3, dim=8
K = torch.randn(2, 5, 8)  # batch=2, key_len=5
V = torch.randn(2, 5, 8)
out, w = scaled_dot_attention(Q, K, V)
print("Output shape:", out.shape)
print("Weights shape:", w.shape)
print("Weights sum to ~1 over keys:", w.sum(dim=-1).round(decimals=3))


Output shape: torch.Size([2, 3, 8])
Weights shape: torch.Size([2, 3, 5])


Weights sum to ~1 over keys: 

tensor([[1., 1., 1.],
        [1., 1., 1.]])


## 8. Single Query: Which Key Is Most Relevant?

A query similar to a key gets high weight.


In [3]:
d = 4
K = torch.tensor([[1., 0, 0, 0],   # key 0: 'cat'
                  [0., 1, 0, 0],   # key 1: 'dog'
                  [0., 0, 1, 0],   # key 2: 'run'
                  [0., 0, 0, 1]])  # key 3: 'eat'

# query close to 'cat'
Q = torch.tensor([[0.9, 0.1, 0.0, 0.0]])
_, w = scaled_dot_attention(Q.unsqueeze(0), K.unsqueeze(0), K.unsqueeze(0))
print("Attention weights:", w.reshape(-1).round(decimals=3))
print("Most weight on key 0 ('cat') -> exactly what we expect.")


Attention weights: tensor([0.3390, 0.2280, 0.2160, 0.2160])
Most weight on key 0 ('cat') -> exactly what we expect.


## 9. Why Divide by sqrt(d_k)?

Without scaling, large dot products push softmax to saturation (all 0 or all 1).


In [4]:
d = 128
Q = torch.randn(5, d)
K = torch.randn(5, d)

raw = Q @ K.t()                       # no scaling
scaled = (Q @ K.t()) / (d ** 0.5)

def entropy(x):
    p = F.softmax(x, dim=-1)
    return -(p * p.log()).sum(dim=-1).mean().item()

print(f"Raw scores   -> softmax entropy: {entropy(raw):.3f} (closer to 0 = saturated)")
print(f"Scaled scores-> softmax entropy: {entropy(scaled):.3f} (higher = more informative)")
print("\nScaling keeps the softmax gradient healthy.")


Raw scores   -> softmax entropy: 0.155 (closer to 0 = saturated)
Scaled scores-> softmax entropy: 1.281 (higher = more informative)

Scaling keeps the softmax gradient healthy.


## 10. Attention as a Module (Bahdanau-style)

A learnable attention module mapping decoder state + encoder outputs to a context vector.


In [5]:
class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.enc = nn.Linear(enc_dim, attn_dim, bias=False)
        self.dec = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, enc_out, dec_hidden):
        # enc_out: (B, T, E), dec_hidden: (B, D)
        B, T, _ = enc_out.shape
        dh = dec_hidden.unsqueeze(1).repeat(1, T, 1)   # (B, T, D)
        energy = torch.tanh(self.enc(enc_out) + self.dec(dh))  # (B, T, A)
        scores = self.v(energy).squeeze(-1)            # (B, T)
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)  # (B, E)
        return context, weights

attn = AdditiveAttention(8, 8, 12)
enc_out = torch.randn(3, 6, 8)
dec_hid = torch.randn(3, 8)
ctx, w = attn(enc_out, dec_hid)
print("Context shape:", ctx.shape)
print("Weights sum to 1:", w.sum(dim=1).round(decimals=3))


Context shape: torch.Size([3, 8])
Weights sum to 1: tensor([1., 1., 1.], grad_fn=<RoundBackward1>)


## 11. Visualize Attention Weights

Attention alignment is a great debugging/interpretability tool.


In [6]:
attn_weights = torch.rand(4, 6)
attn_weights = attn_weights / attn_weights.sum(dim=1, keepdim=True)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(attn_weights.detach().numpy(), cmap='viridis', aspect='auto')
ax.set_xlabel('Source positions (keys)')
ax.set_ylabel('Target steps (queries)')
ax.set_title('Attention alignment heatmap')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('07_07_attention.png', dpi=90)
print("Saved attention heatmap. Rows sum to 1 (each target attends across sources).")


Saved attention heatmap. Rows sum to 1 (each target attends across sources).


## 12. Failure Case: No Masking Attends to Padding

In real sequences, padded positions should be masked or attention wastes weight on them.


In [7]:
B, T, d = 2, 6, 4
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)
Q = torch.randn(B, 1, d)


pad_mask = torch.ones(B, T)
pad_mask[0, 4:] = 0
pad_mask[1, 5:] = 0

_, w_nomask = scaled_dot_attention(Q, K, V)
_, w_mask = scaled_dot_attention(Q, K, V, mask=pad_mask.unsqueeze(1))

print("Without mask, row0 weight on padding:", round(w_nomask[0,0,4:].sum().item(), 3))
print("With mask,    row0 weight on padding:", round(w_mask[0,0,4:].sum().item(), 3))
print("\nMasking zeros out attention to padding -> cleaner context.")


Without mask, row0 weight on padding: 0.321
With mask,    row0 weight on padding: 0.0

Masking zeros out attention to padding -> cleaner context.


## 13. Debugging: Common Errors

- **Uniform weights** — undertrained or wrong scaling. Fix: train longer, add scaling.
- **Softmax all 0/1** — scores too large. Fix: divide by sqrt(d_k).
- **Attends to padding** — no mask. Fix: mask before softmax.
- **Slow training** — full attention matrix. Fix: efficient/truncated attention.

## 14. Real-World Considerations

- Use masked self-attention (no peeking at future) in decoders.
- Multi-head attention (Phase 08) gives richer representation.
- Attention weights are not always linguistically interpretable.

## 15. Common Mistakes

- Confusing self-attention with cross-attention.
- Not scaling scores.
- No masking in decoder.

## 16. When NOT to Use

- Very long sequences without efficient attention implementations.
- When a fixed-context RNN suffices (rare).

## 17. Challenge

Implement simple self-attention (Q,K,V all from the same sequence) and verify it's permutation-equivariant.


In [8]:
# Challenge: self-attention and permutation equivariance
def self_attention(x):
    return scaled_dot_attention(x, x, x)[0]

x = torch.randn(1, 5, 4)
perm = torch.tensor([4, 0, 2, 3, 1])

out = self_attention(x)
out_perm = self_attention(x[:, perm])
equiv = torch.allclose(out[:, perm], out_perm, atol=1e-5)
print("Self-attention is permutation-equivariant:", equiv)
print("This property is why transformers solve NLP with no recurrent order.")


Self-attention is permutation-equivariant: True
This property is why transformers solve NLP with no recurrent order.


## 18. Closed-Book Recall

1. What are Q, K, V?
2. Why divide by sqrt(d_k)?
3. Difference between self-attention and cross-attention?
4. Why mask padding in attention?

## 19. Teach-Back Questions

- Explain attention as 'glancing back' at relevant words.
- Why does attention bridge to transformers?

## 20. Summary

You implemented scaled dot-product and additive attention, saw why scaling matters, visualized weights, and masked padding. Attention is the core of modern NLP.

## 21. Further Experiment

- Add attention to the LSTM classifier; visualize which input tokens matter for each prediction.
- Implement multi-head attention by concatenating multiple attention heads.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
